In [1]:
!pip install PySastrawi


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# =========================================================
# CELL 1 - IMPORT LIBRARY + KONEKSI SUPABASE
# =========================================================
import os
import re
import sys
import pandas as pd
import numpy as np

from pathlib import Path
from dotenv import load_dotenv
from supabase import create_client
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

BACKEND_DIR = next(
    path for path in (Path.cwd(), Path.cwd().parent, Path.cwd() / "backend")
    if (path / "src" / "preprocessing").is_dir()
)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from src.preprocessing.stopwords import get_stopwords

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)


print("✅ Import berhasil")
print("✅ Supabase client siap") 
# =========================================================
# NAMA TABEL SUPABASE
# =========================================================
SOURCE_TABLE = "scholar_article_doi"
CLEANED_TABLE = "cleaned_papers_results"
EVIDENCE_TABLE = "preprocessing_evidence"

print("✅ Nama tabel siap")
print("Source:", SOURCE_TABLE)
print("Cleaned:", CLEANED_TABLE)
print("Evidence:", EVIDENCE_TABLE)

✅ Import berhasil
✅ Supabase client siap
✅ Nama tabel siap
Source: scholar_article_doi
Cleaned: cleaned_papers_results
Evidence: preprocessing_evidence


In [3]:
# =========================================================
# LOAD DATA DARI SUPABASE DOI (AUTO BATCH)
# =========================================================
print("📥 Mengambil data dari Supabase...")

SOURCE_TABLE = "scholar_article_doi"

all_data = []
batch_size = 1000
offset = 0

selected_columns = """
id,
title,
abstract,
authors,
source,
year,
citations,
url,
scraped_at,
scrape_status,
pdf_url,
category,
doi,
pdf_doi,
access_type
"""

while True:
    response = (
        supabase.table(SOURCE_TABLE)
        .select(selected_columns)
        .range(offset, offset + batch_size - 1)
        .execute()
    )

    batch = response.data or []
    if not batch:
        break

    all_data.extend(batch)
    print(f"  → Batch {offset // batch_size + 1}: {len(batch)} data")

    if len(batch) < batch_size:
        break

    offset += batch_size

df = pd.DataFrame(all_data)

print(f"✅ Total data dari {SOURCE_TABLE}: {len(df)} baris")
df.head(3)

📥 Mengambil data dari Supabase...
  → Batch 1: 200 data
✅ Total data dari scholar_article_doi: 200 baris


,id,title,abstract,authors,source,year,citations,url,scraped_at,scrape_status,pdf_url,category,doi,pdf_doi,access_type
0,1,Penerapan machine learning dalam prediksi ting...,… menjabarkan implementasi machine learning un...,"RG Wardhana, G Wang…",Journal of Information …,2023,76,https://jurnal.amikom.ac.id/index.php/joism/ar...,2026-06-28T07:01:10.539972+00:00,pdf_downloaded,https://jurnal.amikom.ac.id/index.php/joism/ar...,machine learning,10.24076/joism.2023v5i1.1136,None,open_access
1,2,Tinjauan Pustaka Sistematis: Penerapan Metode ...,… Machine Learning dapat mempelajari pola data...,"IM Faiza, W Andriani",Jurnal Minfo Polgan,2022,52,https://jurnal.polgan.ac.id/index.php/jmp/arti...,2026-06-28T07:01:17.240487+00:00,pdf_downloaded,https://jurnal.polgan.ac.id/index.php/jmp/arti...,machine learning,10.33395/jmp.v11i2.11657,10.33299/jpkop.22.2.1752,open_access
2,3,"Penerapan Machine Learning, Deep Learning, Dan...","… the application of machine learning, deep le...","S Prasetyo, T Dewayanto",Diponegoro Journal of Accounting,2024,30,https://ejournal3.undip.ac.id/index.php/accoun...,2026-06-28T07:01:22.395722+00:00,pdf_downloaded,https://ejournal3.undip.ac.id/index.php/accoun...,machine learning,10.54373/imeij.v6i6.4112,None,open_access


In [4]:
# =========================================================
# CELL 3 - VALIDASI KOLOM WAJIB
# =========================================================
required_cols = [
    "id",
    "title",
    "abstract",
    "authors",
    "source",
    "year",
    "citations",
    "url",
    "scraped_at",
    "scrape_status",
    "pdf_url",
    "category",
    "doi",
    "pdf_doi",
    "access_type"
]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_cols}")

for col in required_cols:
    df[col] = df[col].fillna("")

df["id"] = df["id"].astype("int64")

print("✅ Kolom wajib valid")
df[required_cols].head(3)

✅ Kolom wajib valid


,id,title,abstract,authors,source,year,citations,url,scraped_at,scrape_status,pdf_url,category,doi,pdf_doi,access_type
0,1,Penerapan machine learning dalam prediksi ting...,… menjabarkan implementasi machine learning un...,"RG Wardhana, G Wang…",Journal of Information …,2023,76,https://jurnal.amikom.ac.id/index.php/joism/ar...,2026-06-28T07:01:10.539972+00:00,pdf_downloaded,https://jurnal.amikom.ac.id/index.php/joism/ar...,machine learning,10.24076/joism.2023v5i1.1136,,open_access
1,2,Tinjauan Pustaka Sistematis: Penerapan Metode ...,… Machine Learning dapat mempelajari pola data...,"IM Faiza, W Andriani",Jurnal Minfo Polgan,2022,52,https://jurnal.polgan.ac.id/index.php/jmp/arti...,2026-06-28T07:01:17.240487+00:00,pdf_downloaded,https://jurnal.polgan.ac.id/index.php/jmp/arti...,machine learning,10.33395/jmp.v11i2.11657,10.33299/jpkop.22.2.1752,open_access
2,3,"Penerapan Machine Learning, Deep Learning, Dan...","… the application of machine learning, deep le...","S Prasetyo, T Dewayanto",Diponegoro Journal of Accounting,2024,30,https://ejournal3.undip.ac.id/index.php/accoun...,2026-06-28T07:01:22.395722+00:00,pdf_downloaded,https://ejournal3.undip.ac.id/index.php/accoun...,machine learning,10.54373/imeij.v6i6.4112,,open_access


In [5]:
# =========================================================
# CELL 4 - STOPWORDS + STEMMER
# =========================================================
stop_words = get_stopwords()

stemmer = StemmerFactory().create_stemmer()

print("✅ Stopwords dan stemmer siap")

✅ Stopwords dan stemmer siap


In [6]:
# =========================================================
# CELL 5 - FUNGSI PREPROCESSING
# =========================================================
def safe_text(value):
    if value is None:
        return ""

    if isinstance(value, list):
        return " ".join(str(item) for item in value)

    if isinstance(value, dict):
        return " ".join(str(item) for item in value.values())

    return str(value)

def clean_text(text):
    text = safe_text(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"doi\s*[:/]?\s*", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenizing(text):
    if not isinstance(text, str):
        return []
    return text.split()

def remove_stopwords(tokens):
    if not isinstance(tokens, list):
        return []

    return [
        token for token in tokens
        if token not in stop_words and len(token) > 1
    ]

def stemming(tokens):
    if not isinstance(tokens, list):
        return []

    return [stemmer.stem(token) for token in tokens]

print("✅ Fungsi preprocessing siap")

✅ Fungsi preprocessing siap


In [7]:
# =========================================================
# CELL 6 - GABUNGKAN SEMUA METADATA MENJADI FULL_TEXT
# =========================================================
metadata_cols = [
    "title",
    "abstract",
    "authors",
    "source",
    "year",
    "citations",
    "url",
    "scraped_at",
    "scrape_status",
    "pdf_url",
    "category",
    "doi",
    "pdf_doi",
    "access_type"
]

df["full_text"] = df[metadata_cols].astype(str).agg(" ".join, axis=1).str.strip()

print("✅ full_text dibuat dari semua metadata")
df[["id", "title", "full_text"]].head(3)

✅ full_text dibuat dari semua metadata


,id,title,full_text
0,1,Penerapan machine learning dalam prediksi ting...,Penerapan machine learning dalam prediksi ting...
1,2,Tinjauan Pustaka Sistematis: Penerapan Metode ...,Tinjauan Pustaka Sistematis: Penerapan Metode ...
2,3,"Penerapan Machine Learning, Deep Learning, Dan...","Penerapan Machine Learning, Deep Learning, Dan..."


In [8]:
# =========================================================
# CELL 7 - STEP 1 TEXT CLEANSING
# =========================================================
df["cleaned"] = df["full_text"].apply(clean_text)

print("✅ Step 1 selesai: Text Cleansing")
df[["full_text", "cleaned"]].head(3)

✅ Step 1 selesai: Text Cleansing


,full_text,cleaned
0,Penerapan machine learning dalam prediksi ting...,penerapan machine learning dalam prediksi ting...
1,Tinjauan Pustaka Sistematis: Penerapan Metode ...,tinjauan pustaka sistematis penerapan metode m...
2,"Penerapan Machine Learning, Deep Learning, Dan...",penerapan machine learning deep learning dan d...


In [9]:
# =========================================================
# CELL 8 - STEP 2 TOKENIZATION
# =========================================================
df["tokens"] = df["cleaned"].apply(tokenizing)

print("✅ Step 2 selesai: Tokenization")
df[["cleaned", "tokens"]].head(3)

✅ Step 2 selesai: Tokenization


,cleaned,tokens
0,penerapan machine learning dalam prediksi ting...,"[penerapan, machine, learning, dalam, prediksi..."
1,tinjauan pustaka sistematis penerapan metode m...,"[tinjauan, pustaka, sistematis, penerapan, met..."
2,penerapan machine learning deep learning dan d...,"[penerapan, machine, learning, deep, learning,..."


In [10]:
# =========================================================
# CELL 9 - STEP 3 STOPWORD REMOVAL
# =========================================================
df["tokens_clean"] = df["tokens"].apply(remove_stopwords)

print("✅ Step 3 selesai: Stopword Removal")
df[["tokens", "tokens_clean"]].head(3)

✅ Step 3 selesai: Stopword Removal


,tokens,tokens_clean
0,"[penerapan, machine, learning, dalam, prediksi...","[penerapan, machine, learning, prediksi, tingk..."
1,"[tinjauan, pustaka, sistematis, penerapan, met...","[tinjauan, pustaka, sistematis, penerapan, met..."
2,"[penerapan, machine, learning, deep, learning,...","[penerapan, machine, learning, deep, learning,..."


In [11]:
# =========================================================
# STEP 10 - STEMMING (Sastrawi / Nazief-Adriani)
# =========================================================
df["tokens_stemmed"] = df["tokens_clean"].apply(stemming)

print("✅ Step 4 selesai: Stemming")
df[["tokens_clean", "tokens_stemmed"]].head(3)

✅ Step 4 selesai: Stemming


,tokens_clean,tokens_stemmed
0,"[penerapan, machine, learning, prediksi, tingk...","[terap, machine, learning, prediksi, tingkat, ..."
1,"[tinjauan, pustaka, sistematis, penerapan, met...","[tinjau, pustaka, sistematis, terap, metode, m..."
2,"[penerapan, machine, learning, deep, learning,...","[terap, machine, learning, deep, learning, dat..."


In [12]:
# =========================================================
# CELL 11 - GABUNGKAN TOKEN MENJADI CLEANED_TEXT
# =========================================================
df["cleaned_text"] = df["tokens_stemmed"].apply(
    lambda tokens: " ".join(tokens) if isinstance(tokens, list) else ""
)

print("✅ cleaned_text selesai dibuat")
print(f"📊 Total baris diproses: {len(df)}")
print(f"📊 cleaned_text kosong: {int(df['cleaned_text'].eq('').sum())}")

df[["title", "cleaned_text"]].head(5)

✅ cleaned_text selesai dibuat
📊 Total baris diproses: 200
📊 cleaned_text kosong: 0


,title,cleaned_text
0,Penerapan machine learning dalam prediksi ting...,terap machine learning prediksi tingkat kasus ...
1,Tinjauan Pustaka Sistematis: Penerapan Metode ...,tinjau pustaka sistematis terap metode machine...
2,"Penerapan Machine Learning, Deep Learning, Dan...",terap machine learning deep learning data mini...
3,Penerapan machine learning untuk prediksi benc...,terap machine learning prediksi bencana banjir...
4,Penerapan Machine Learning dan Deep Learning p...,terap machine learning deep learning tingkat d...


In [13]:
# =========================================================
# AUDIT KUALITAS PREPROCESSING
# =========================================================
df["len_full_text"] = df["full_text"].apply(lambda x: len(str(x).split()))
df["len_cleaned_text"] = df["cleaned_text"].apply(lambda x: len(str(x).split()))

print("Rata-rata token sebelum preprocessing:", round(df["len_full_text"].mean(), 2))
print("Rata-rata token sesudah preprocessing:", round(df["len_cleaned_text"].mean(), 2))
print("Dokumen jadi kosong setelah preprocessing:", int((df["len_cleaned_text"] == 0).sum()))



Rata-rata token sebelum preprocessing: 58.04
Rata-rata token sesudah preprocessing: 54.96
Dokumen jadi kosong setelah preprocessing: 0


In [14]:
# =========================================================
# CELL 13 - CONTOH DATA ACAK
# =========================================================
sample_cols = [
    "title",
    "full_text",
    "cleaned",
    "tokens",
    "tokens_clean",
    "tokens_stemmed",
    "cleaned_text"
]

df[sample_cols].sample(min(5, len(df)), random_state=42)

,title,full_text,cleaned,tokens,tokens_clean,tokens_stemmed,cleaned_text
95,Artificial intelligence in cyber security,Artificial intelligence in cyber security … Fo...,artificial intelligence in cyber security for ...,"[artificial, intelligence, in, cyber, security...","[artificial, intelligence, cyber, security, se...","[artificial, intelligence, cyber, security, se...",artificial intelligence cyber security securit...
15,Penerapan Machine Learning menggunakan algorit...,Penerapan Machine Learning menggunakan algorit...,penerapan machine learning menggunakan algorit...,"[penerapan, machine, learning, menggunakan, al...","[penerapan, machine, learning, menggunakan, al...","[terap, machine, learning, guna, algoritma, c4...",terap machine learning guna algoritma c4 bas p...
30,Financial machine learning,Financial machine learning … We survey the nas...,financial machine learning we survey the nasce...,"[financial, machine, learning, we, survey, the...","[financial, machine, learning, we, survey, nas...","[financial, machine, learning, we, survey, nas...",financial machine learning we survey nascent l...
158,Perancangan Aplikasi Pemesanan Makanan Berbasi...,Perancangan Aplikasi Pemesanan Makanan Berbasi...,perancangan aplikasi pemesanan makanan berbasi...,"[perancangan, aplikasi, pemesanan, makanan, be...","[perancangan, aplikasi, pemesanan, makanan, be...","[ancang, aplikasi, mesan, makan, bas, web, bas...",ancang aplikasi mesan makan bas web bas web ap...
128,A study of mobile app use for teaching and res...,A study of mobile app use for teaching and res...,a study of mobile app use for teaching and res...,"[a, study, of, mobile, app, use, for, teaching...","[study, mobile, app, use, teaching, research, ...","[study, mobile, app, use, teaching, research, ...",study mobile app use teaching research higher ...


In [15]:
# =========================================================
# CELL 14 - HELPER UPSERT SUPABASE
# =========================================================
def to_records_safe(dataframe: pd.DataFrame):
    safe_df = dataframe.replace({np.nan: None})
    return safe_df.to_dict(orient="records")

def upsert_batches(table_name: str, records: list, on_conflict: str, batch_size: int = 500):
    total = len(records)

    if total == 0:
        print(f"⚠️ Tidak ada data untuk di-upsert ke {table_name}")
        return

    for i in range(0, total, batch_size):
        batch = records[i:i + batch_size]

        supabase.table(table_name).upsert(
            batch,
            on_conflict=on_conflict
        ).execute()

        print(f"  → Batch {i // batch_size + 1}: {len(batch)} data")

    print(f"✅ Upsert {total} baris ke {table_name}")

In [16]:
# =========================================================
# IMPORT TAMBAHAN UNTUK TIMESTAMP
# =========================================================
from datetime import datetime, timezone

ts = datetime.now(timezone.utc).isoformat()
print("Timestamp:", ts)
# =========================================================
# CELL 16 - SIMPAN HASIL CLEANED (CSV + SUPABASE)
# target table: cleaned_papers_results
# =========================================================
base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
save_dir = os.path.join(base_dir, "data")
save_path = os.path.join(save_dir, "cleaned_papers.csv")
os.makedirs(save_dir, exist_ok=True)

save_df = df[
    ["id", "title", "abstract", "authors", "year", "source", "category", "pdf_url", "url", "scrape_status", "cleaned_text"]
].copy()

save_df["updated_at"] = ts
save_df["id"] = save_df["id"].astype("int64")

save_df.to_csv(save_path, index=False)

print(f"✅ CSV cleaned diperbarui: {save_path}")
print(f"📊 Total baris CSV: {len(save_df)}")

upsert_batches(
    table_name=CLEANED_TABLE,
    records=to_records_safe(save_df),
    on_conflict="id",
    batch_size=500
)

save_df.head(5)

Timestamp: 2026-06-28T10:40:47.030684+00:00
✅ CSV cleaned diperbarui: d:\Tugas Akhir\paperci_artikel\backend\data\cleaned_papers.csv
📊 Total baris CSV: 200
  → Batch 1: 200 data
✅ Upsert 200 baris ke cleaned_papers_results


,id,title,abstract,authors,year,source,category,pdf_url,url,scrape_status,cleaned_text,updated_at
0,1,Penerapan machine learning dalam prediksi ting...,… menjabarkan implementasi machine learning un...,"RG Wardhana, G Wang…",2023,Journal of Information …,machine learning,https://jurnal.amikom.ac.id/index.php/joism/ar...,https://jurnal.amikom.ac.id/index.php/joism/ar...,pdf_downloaded,terap machine learning prediksi tingkat kasus ...,2026-06-28T10:40:47.030684+00:00
1,2,Tinjauan Pustaka Sistematis: Penerapan Metode ...,… Machine Learning dapat mempelajari pola data...,"IM Faiza, W Andriani",2022,Jurnal Minfo Polgan,machine learning,https://jurnal.polgan.ac.id/index.php/jmp/arti...,https://jurnal.polgan.ac.id/index.php/jmp/arti...,pdf_downloaded,tinjau pustaka sistematis terap metode machine...,2026-06-28T10:40:47.030684+00:00
2,3,"Penerapan Machine Learning, Deep Learning, Dan...","… the application of machine learning, deep le...","S Prasetyo, T Dewayanto",2024,Diponegoro Journal of Accounting,machine learning,https://ejournal3.undip.ac.id/index.php/accoun...,https://ejournal3.undip.ac.id/index.php/accoun...,pdf_downloaded,terap machine learning deep learning data mini...,2026-06-28T10:40:47.030684+00:00
3,4,Penerapan machine learning untuk prediksi benc...,Indonesia beriklim tropis karena terletak pada...,"E Pitaloka, TBA Hartanto, S Sandiwarno",2024,J. Sist. Inf. Bisnis,machine learning,https://scholar.archive.org/work/qzvxkp6mgrfnn...,https://scholar.archive.org/work/qzvxkp6mgrfnn...,pdf_failed,terap machine learning prediksi bencana banjir...,2026-06-28T10:40:47.030684+00:00
4,5,Penerapan Machine Learning dan Deep Learning p...,… This research aims to explore the applicatio...,"BV Tarissa, T Dewayanto",2024,Diponegoro Journal of Accounting,machine learning,https://ejournal3.undip.ac.id/index.php/accoun...,https://ejournal3.undip.ac.id/index.php/accoun...,pdf_downloaded,terap machine learning deep learning tingkat d...,2026-06-28T10:40:47.030684+00:00


In [17]:
# =========================================================
# CELL 17 - SIMPAN EVIDENCE PREPROCESSING
# target table: preprocessing_evidence
# =========================================================
evidence_path = os.path.join(save_dir, "preprocessing_evidence_doi_sample.csv")

evidence_cols = [
    "id",
    "title",
    "full_text",
    "cleaned",
    "tokens",
    "tokens_clean",
    "tokens_stemmed",
    "cleaned_text"
]

evidence_df = df[evidence_cols].copy()

evidence_df["updated_at"] = ts
evidence_df["id"] = evidence_df["id"].astype("int64")

evidence_df.head(50).to_csv(evidence_path, index=False)

print(f"✅ Evidence sample CSV diperbarui: {evidence_path}")

upsert_batches(
    table_name=EVIDENCE_TABLE,
    records=to_records_safe(evidence_df),
    on_conflict="id",
    batch_size=500
)

print(f"📊 Total baris evidence ke Supabase: {len(evidence_df)}")
evidence_df.head(5)

✅ Evidence sample CSV diperbarui: d:\Tugas Akhir\paperci_artikel\backend\data\preprocessing_evidence_doi_sample.csv
  → Batch 1: 200 data
✅ Upsert 200 baris ke preprocessing_evidence
📊 Total baris evidence ke Supabase: 200


,id,title,full_text,cleaned,tokens,tokens_clean,tokens_stemmed,cleaned_text,updated_at
0,1,Penerapan machine learning dalam prediksi ting...,Penerapan machine learning dalam prediksi ting...,penerapan machine learning dalam prediksi ting...,"[penerapan, machine, learning, dalam, prediksi...","[penerapan, machine, learning, prediksi, tingk...","[terap, machine, learning, prediksi, tingkat, ...",terap machine learning prediksi tingkat kasus ...,2026-06-28T10:40:47.030684+00:00
1,2,Tinjauan Pustaka Sistematis: Penerapan Metode ...,Tinjauan Pustaka Sistematis: Penerapan Metode ...,tinjauan pustaka sistematis penerapan metode m...,"[tinjauan, pustaka, sistematis, penerapan, met...","[tinjauan, pustaka, sistematis, penerapan, met...","[tinjau, pustaka, sistematis, terap, metode, m...",tinjau pustaka sistematis terap metode machine...,2026-06-28T10:40:47.030684+00:00
2,3,"Penerapan Machine Learning, Deep Learning, Dan...","Penerapan Machine Learning, Deep Learning, Dan...",penerapan machine learning deep learning dan d...,"[penerapan, machine, learning, deep, learning,...","[penerapan, machine, learning, deep, learning,...","[terap, machine, learning, deep, learning, dat...",terap machine learning deep learning data mini...,2026-06-28T10:40:47.030684+00:00
3,4,Penerapan machine learning untuk prediksi benc...,Penerapan machine learning untuk prediksi benc...,penerapan machine learning untuk prediksi benc...,"[penerapan, machine, learning, untuk, prediksi...","[penerapan, machine, learning, prediksi, benca...","[terap, machine, learning, prediksi, bencana, ...",terap machine learning prediksi bencana banjir...,2026-06-28T10:40:47.030684+00:00
4,5,Penerapan Machine Learning dan Deep Learning p...,Penerapan Machine Learning dan Deep Learning p...,penerapan machine learning dan deep learning p...,"[penerapan, machine, learning, dan, deep, lear...","[penerapan, machine, learning, deep, learning,...","[terap, machine, learning, deep, learning, tin...",terap machine learning deep learning tingkat d...,2026-06-28T10:40:47.030684+00:00


In [18]:
# =========================================================
# CELL 18 - VALIDASI CEPAT SUPABASE
# =========================================================
check_cleaned = (
    supabase.table(CLEANED_TABLE)
    .select("id", count="exact")
    .limit(1)
    .execute()
)

check_evidence = (
    supabase.table(EVIDENCE_TABLE)
    .select("id", count="exact")
    .limit(1)
    .execute()
)

print("✅ cleaned_papers_results count:", check_cleaned.count)
print("✅ preprocessing_evidence count:", check_evidence.count)

✅ cleaned_papers_results count: 200
✅ preprocessing_evidence count: 200
